# 동적 웹페이지 크롤링 : Naver News Infinite Scroll

네이버 뉴스 검색 결과가 무한 스크롤로 추가되는 동안 **새로 나타난 메인 기사만 즉시 수집**하고,
수집 종료 후 이미지를 다운로드한 뒤 메타데이터 CSV를 저장한다.

```text
네이버 뉴스 검색
→ 현재 DOM의 새 기사 수집
→ 아래로 스크롤
→ 새 기사 로딩 대기
→ 새 기사 수집
→ 종료 조건 확인
→ 이미지 다운로드
→ CSV 저장 및 검증
```

## 종료 조건

다음 조건 중 하나를 만족하면 수집을 종료한다.

1. 목표 기사 수(`MAX_ARTICLES`)에 도달
2. 연속 `MAX_IDLE_SCROLLS`회 동안 새로운 기사 없음
3. 안전 제한인 `MAX_SCROLLS`회에 도달

> 무한 스크롤에서는 먼저 끝까지 스크롤한 뒤 한 번에 파싱하기보다,
> 스크롤할 때마다 새로 로딩된 데이터를 수집하는 방식이 DOM 변경과 메모리 사용에 더 안전하다.


# 라이브러리

In [1]:
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlparse

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from selenium import webdriver
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from urllib3.util.retry import Retry

# 기본 설정

In [2]:
TARGET_URL = 'https://www.naver.com/'
SEARCH_KEYWORD = 'AI'

WAIT_TIMEOUT = 10
SCROLL_WAIT_TIMEOUT = 5

## 수집 종료 설정
MAX_ARTICLES = 100
MAX_SCROLLS = 30
MAX_IDLE_SCROLLS = 3

## False : 브라우저 화면 표시
## True : 브라우저 화면을 표시하지 않고 실행
HEADLESS = False

CONNECT_TIMEOUT = 10
READ_TIMEOUT = 30

PROJECT_DIR = Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic' / 'naver'

TITLE_SELECTOR = (
    'a[data-heatmap-target=".tit"] '
    '> span.sds-comps-text-type-headline1'
)
PRESS_SELECTOR = (
    '.sds-comps-profile-info-title-text '
    'a[href*="media.naver.com/press/"] '
    'span.sds-comps-text'
)
PRESS_FALLBACK_SELECTOR = '.sds-comps-profile-info-title-text'
SUMMARY_SELECTOR = (
    'a[data-heatmap-target=".body"] '
    'span.sds-comps-text-type-body1'
)
IMAGE_SELECTOR = 'a[data-heatmap-target=".img"] img'

NEWS_CARD_XPATH = './ancestor::div[.//*[@data-sds-comp="Profile"]][1]'

## 수집량 권장값

일반적인 분석 실습에서는 `MAX_ARTICLES = 100` 정도가 적당

- 너무 적으면 무한 스크롤 수집의 의미가 약함
- 너무 많으면 이미지 다운로드 시간과 저장 용량이 커짐
- 결과 재현성을 위해 **스크롤 횟수보다 기사 수를 주 종료 기준**으로 사용
- `MAX_SCROLLS`, `MAX_IDLE_SCROLLS`는 사이트 응답 이상에 대비한 안전장치


# WebDriver 및 HTTP Session

In [3]:
def create_driver(headless: bool) -> webdriver.Chrome:
    """Chrome WebDriver를 생성하여 반환한다."""

    options = Options()

    if headless:
        options.add_argument('--headless=new')

    options.add_argument('--start-maximized')

    return webdriver.Chrome(options=options)


def create_http_session() -> requests.Session:
    """이미지 다운로드용 HTTP Session을 생성한다."""

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({'GET'}),
        respect_retry_after_header=True,
    )

    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))
    session.headers.update({
        'User-Agent': 'EducationalDataCollector/1.0',
    })

    return session

# 네이버 뉴스 검색

In [4]:
def open_naver_news_search(
    driver: webdriver.Chrome,
    search_keyword: str,
    wait_timeout: int,
) -> str:
    """네이버에서 검색어를 입력하고 뉴스 검색 결과로 이동한다."""

    wait = WebDriverWait(driver, wait_timeout)

    driver.get(TARGET_URL)

    query = wait.until(
        EC.element_to_be_clickable((By.ID, 'query'))
    )
    query.clear()
    query.send_keys(search_keyword)
    query.send_keys(Keys.ENTER)

    news_tab = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//div[@id='lnb']//a[@role='tab' and normalize-space()='뉴스']",
            )
        )
    )
    news_tab.click()

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, TITLE_SELECTOR)
        )
    )

    return driver.current_url

# 파일 및 배치 경로

In [5]:
def sanitize_name(value: str, max_length: int = 80) -> str:
    """폴더명 또는 파일명으로 사용할 문자열을 정리한다."""

    value = re.sub(r'[\\/:*?"<>|]', '_', value)
    value = re.sub(r'\s+', ' ', value).strip()
    value = value.rstrip('. ')

    return (value or 'untitled')[:max_length]


def create_batch_directory(
    output_dir: Path,
    search_keyword: str,
    batch_id: str,
) -> tuple[Path, Path]:
    """검색어와 배치 ID를 기준으로 저장 폴더를 생성한다."""

    keyword_name = sanitize_name(search_keyword)

    batch_dir = output_dir / keyword_name / batch_id
    image_dir = batch_dir / 'images'

    image_dir.mkdir(parents=True, exist_ok=True)

    return batch_dir, image_dir

# 뉴스 한 건 파싱

In [6]:
def get_original_image_url(image_url: str) -> str:
    """네이버 이미지 프록시 URL에서 원본 src URL을 추출한다."""

    if not image_url:
        return ''

    parsed = urlparse(image_url)
    source_values = parse_qs(parsed.query).get('src')

    if not source_values:
        return image_url

    return unquote(source_values[0])


def extract_press(news_card: WebElement) -> str:
    """뉴스 카드에서 언론사명을 추출한다."""

    press_elements = news_card.find_elements(
        By.CSS_SELECTOR,
        PRESS_SELECTOR,
    )

    if press_elements:
        return press_elements[0].text.strip()

    fallback_elements = news_card.find_elements(
        By.CSS_SELECTOR,
        PRESS_FALLBACK_SELECTOR,
    )

    if fallback_elements:
        return fallback_elements[0].text.splitlines()[0].strip()

    return ''


def parse_news_item(
    title_element: WebElement,
    rank: int,
    scroll_round: int,
    search_keyword: str,
    batch_id: str,
    source_url: str,
) -> dict[str, str | int]:
    """메인 기사 제목 요소를 기준으로 뉴스 한 건을 파싱한다."""

    news_card = title_element.find_element(
        By.XPATH,
        NEWS_CARD_XPATH,
    )

    title = title_element.text.strip()

    title_link = title_element.find_element(By.XPATH, '..')
    article_url = title_link.get_attribute('href') or ''

    summary_elements = news_card.find_elements(
        By.CSS_SELECTOR,
        SUMMARY_SELECTOR,
    )
    summary = summary_elements[0].text.strip() if summary_elements else ''

    image_elements = news_card.find_elements(
        By.CSS_SELECTOR,
        IMAGE_SELECTOR,
    )

    image_url = ''

    if image_elements:
        image_url = (
            image_elements[0].get_property('currentSrc')
            or image_elements[0].get_attribute('src')
            or ''
        )

    return {
        'rank': rank,
        'scroll_round': scroll_round,
        'batch_id': batch_id,
        'search_keyword': search_keyword,
        'press': extract_press(news_card),
        'title': title,
        'summary': summary,
        'article_url': article_url,
        'image_url': image_url,
        'image_source_url': get_original_image_url(image_url),
        'source_site': 'Naver News Search',
        'source_url': source_url,
        'collected_at': datetime.now().isoformat(timespec='seconds'),
    }

# 무한 스크롤 수집

In [7]:
def has_unseen_news(
    driver: webdriver.Chrome,
    seen_article_urls: set[str],
) -> bool:
    """현재 DOM에 아직 수집하지 않은 메인 기사가 있는지 확인한다."""

    title_elements = driver.find_elements(
        By.CSS_SELECTOR,
        TITLE_SELECTOR,
    )

    for title_element in title_elements:
        try:
            article_url = (
                title_element
                .find_element(By.XPATH, '..')
                .get_attribute('href')
            )

        except StaleElementReferenceException:
            continue

        if article_url and article_url not in seen_article_urls:
            return True

    return False


def scroll_and_wait(
    driver: webdriver.Chrome,
    seen_article_urls: set[str],
    wait_timeout: int,
) -> bool:
    """페이지 하단으로 스크롤하고 새로운 기사가 나타날 때까지 기다린다."""

    driver.execute_script(
        'window.scrollTo(0, document.documentElement.scrollHeight);'
    )

    try:
        WebDriverWait(driver, wait_timeout).until(
            lambda current_driver: has_unseen_news(
                current_driver,
                seen_article_urls,
            )
        )
        return True

    except TimeoutException:
        return False

In [8]:
def collect_news_with_infinite_scroll(
    driver: webdriver.Chrome,
    search_keyword: str,
    batch_id: str,
    max_articles: int | None,
    max_scrolls: int,
    max_idle_scrolls: int,
    scroll_wait_timeout: int,
) -> tuple[pd.DataFrame, dict[str, int | str]]:
    """
    무한 스크롤을 수행하면서 새로 로딩된 메인 뉴스만 수집한다.

    종료 조건:
        - max_articles 도달
        - 연속 max_idle_scrolls회 동안 새 기사 없음
        - max_scrolls 도달
    """

    source_url = driver.current_url
    news_records = []
    seen_article_urls: set[str] = set()

    scroll_count = 0
    idle_scrolls = 0
    stop_reason = ''

    while True:
        title_elements = driver.find_elements(
            By.CSS_SELECTOR,
            TITLE_SELECTOR,
        )

        before_count = len(news_records)

        for title_element in title_elements:
            try:
                article_url = (
                    title_element
                    .find_element(By.XPATH, '..')
                    .get_attribute('href')
                ) or ''

                if not article_url or article_url in seen_article_urls:
                    continue

                rank = len(news_records) + 1

                news = parse_news_item(
                    title_element=title_element,
                    rank=rank,
                    scroll_round=scroll_count,
                    search_keyword=search_keyword,
                    batch_id=batch_id,
                    source_url=source_url,
                )

            except (NoSuchElementException, StaleElementReferenceException) as error:
                print(
                    f'[scroll={scroll_count:02d}] '
                    f'기사 추출 실패 : {error.__class__.__name__}'
                )
                continue

            seen_article_urls.add(article_url)
            news_records.append(news)

            if max_articles is not None and len(news_records) >= max_articles:
                break

        new_count = len(news_records) - before_count

        print(
            f'[scroll={scroll_count:02d}] '
            f'신규 {new_count:3d}건 / 누적 {len(news_records):3d}건'
        )

        if max_articles is not None and len(news_records) >= max_articles:
            stop_reason = f'목표 기사 수 {max_articles}건 도달'
            break

        if scroll_count >= max_scrolls:
            stop_reason = f'최대 스크롤 횟수 {max_scrolls}회 도달'
            break

        if scroll_count > 0:
            if new_count == 0:
                idle_scrolls += 1
            else:
                idle_scrolls = 0

        if idle_scrolls >= max_idle_scrolls:
            stop_reason = (
                f'연속 {max_idle_scrolls}회 스크롤 동안 '
                '새로운 기사 없음'
            )
            break

        scroll_count += 1

        loaded = scroll_and_wait(
            driver=driver,
            seen_article_urls=seen_article_urls,
            wait_timeout=scroll_wait_timeout,
        )

        if not loaded:
            print(
                f'[scroll={scroll_count:02d}] '
                '대기 시간 안에 새 기사를 확인하지 못했습니다.'
            )

    news_df = pd.DataFrame(news_records)

    if news_df.empty:
        raise ValueError('수집된 뉴스 기사가 없습니다.')

    if news_df['article_url'].duplicated().any():
        raise ValueError('중복된 기사 URL이 존재합니다.')

    collection_summary = {
        'article_count': len(news_df),
        'scroll_count': scroll_count,
        'idle_scrolls': idle_scrolls,
        'stop_reason': stop_reason,
    }

    return news_df, collection_summary

### 왜 스크롤하면서 바로 수집하는가?

무한 스크롤 페이지는 아래쪽 콘텐츠를 계속 추가하거나, 경우에 따라 오래된 DOM 요소를 교체할 수 있다.

따라서:

```text
끝까지 모두 스크롤
→ 마지막에 한 번에 추출
```

보다:

```text
현재 데이터 추출
→ URL 기준 중복 제거
→ 스크롤
→ 새 데이터 추출
```

방식이 안정적이다.


# 이미지 다운로드

In [9]:
def get_image_extension(response: requests.Response) -> str:
    """응답 Content-Type을 기준으로 이미지 확장자를 반환한다."""

    content_type = response.headers.get('Content-Type', '').split(';')[0].lower()

    extension_map = {
        'image/jpeg': '.jpg',
        'image/png': '.png',
        'image/webp': '.webp',
        'image/gif': '.gif',
    }

    return extension_map.get(content_type, '.jpg')


def download_image(
    session: requests.Session,
    image_urls: list[str],
    image_dir: Path,
    rank: int,
    title: str,
) -> Path:
    """후보 URL을 순서대로 요청하여 이미지를 저장한다."""

    last_error = None

    for image_url in dict.fromkeys(url for url in image_urls if url):
        try:
            response = session.get(
                image_url,
                timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
            )
            response.raise_for_status()

            content_type = response.headers.get('Content-Type', '').lower()

            if not content_type.startswith('image/'):
                raise ValueError(
                    f'이미지 응답이 아닙니다. Content-Type={content_type}'
                )

            extension = get_image_extension(response)
            safe_title = sanitize_name(title)

            image_file = image_dir / f'{rank:03d}_{safe_title}{extension}'
            temp_file = image_file.with_suffix(image_file.suffix + '.part')

            temp_file.write_bytes(response.content)
            temp_file.replace(image_file)

            return image_file

        except (requests.exceptions.RequestException, ValueError) as error:
            last_error = error

    if last_error is not None:
        raise last_error

    raise ValueError('다운로드할 이미지 URL이 없습니다.')

In [10]:
def download_news_images(
    news_df: pd.DataFrame,
    image_dir: Path,
    project_dir: Path,
) -> pd.DataFrame:
    """뉴스 이미지를 저장하고 로컬 파일 경로와 성공 여부를 추가한다."""

    result_df = news_df.copy()

    result_df['image_file'] = ''
    result_df['image_downloaded'] = False

    session = create_http_session()

    try:
        for index, row in result_df.iterrows():
            image_urls = [
                row['image_source_url'],
                row['image_url'],
            ]

            if not any(image_urls):
                continue

            try:
                image_file = download_image(
                    session=session,
                    image_urls=image_urls,
                    image_dir=image_dir,
                    rank=int(row['rank']),
                    title=row['title'],
                )

            except (requests.exceptions.RequestException, ValueError) as error:
                print(
                    f'[{int(row["rank"]):03d}] 이미지 저장 실패 '
                    f': {error.__class__.__name__}'
                )
                continue

            result_df.at[index, 'image_file'] = str(
                image_file.relative_to(project_dir)
            )
            result_df.at[index, 'image_downloaded'] = True

    finally:
        session.close()

    return result_df

# CSV 저장 및 검증

In [11]:
NEWS_COLUMNS = [
    'rank',
    'scroll_round',
    'batch_id',
    'search_keyword',
    'press',
    'title',
    'summary',
    'article_url',
    'image_url',
    'image_source_url',
    'image_file',
    'image_downloaded',
    'source_site',
    'source_url',
    'collected_at',
]


def validate_news_dataframe(news_df: pd.DataFrame) -> None:
    """CSV 저장 전에 필수 컬럼과 기사 URL 중복을 검증한다."""

    missing_columns = set(NEWS_COLUMNS) - set(news_df.columns)

    if missing_columns:
        raise ValueError(
            f'필수 컬럼이 누락되었습니다. {sorted(missing_columns)}'
        )

    required_columns = [
        'rank',
        'batch_id',
        'search_keyword',
        'title',
        'article_url',
        'source_url',
        'collected_at',
    ]

    if news_df[required_columns].isna().any().any():
        raise ValueError('필수 데이터에 결측값이 존재합니다.')

    if news_df['article_url'].duplicated().any():
        raise ValueError('중복된 기사 URL이 존재합니다.')


def save_news_csv(
    news_df: pd.DataFrame,
    csv_file: Path,
) -> Path:
    """뉴스 DataFrame을 임시 파일을 거쳐 CSV로 저장한다."""

    validate_news_dataframe(news_df)

    csv_file.parent.mkdir(parents=True, exist_ok=True)
    temp_file = csv_file.with_suffix('.tmp')

    news_df[NEWS_COLUMNS].to_csv(
        temp_file,
        index=False,
        encoding='utf-8-sig',
    )

    temp_file.replace(csv_file)

    return csv_file


def verify_saved_csv(
    csv_file: Path,
    original_df: pd.DataFrame,
) -> pd.DataFrame:
    """저장한 CSV를 다시 읽어 행 수와 기사 URL을 검증한다."""

    saved_df = pd.read_csv(csv_file)

    if len(saved_df) != len(original_df):
        raise ValueError('CSV 저장 전후의 행 수가 다릅니다.')

    if saved_df['article_url'].tolist() != original_df['article_url'].tolist():
        raise ValueError('CSV 저장 전후의 기사 URL 순서가 다릅니다.')

    return saved_df

# 전체 실행

In [12]:
def run_naver_news_collection(
    search_keyword: str = SEARCH_KEYWORD,
    max_articles: int | None = MAX_ARTICLES,
    max_scrolls: int = MAX_SCROLLS,
    max_idle_scrolls: int = MAX_IDLE_SCROLLS,
    headless: bool = HEADLESS,
    wait_timeout: int = WAIT_TIMEOUT,
    scroll_wait_timeout: int = SCROLL_WAIT_TIMEOUT,
) -> tuple[pd.DataFrame, Path, Path, dict[str, int | str]]:
    """네이버 뉴스 무한 스크롤 수집부터 이미지·CSV 저장까지 실행한다."""

    batch_started_at = datetime.now()
    batch_id = batch_started_at.strftime('%Y%m%d_%H%M%S')

    batch_dir, image_dir = create_batch_directory(
        output_dir=OUTPUT_DIR,
        search_keyword=search_keyword,
        batch_id=batch_id,
    )

    keyword_name = sanitize_name(search_keyword)
    csv_file = batch_dir / f'naver_news_{keyword_name}_{batch_id}.csv'

    driver = create_driver(headless)
    news_df = None
    collection_summary = {}

    try:
        source_url = open_naver_news_search(
            driver=driver,
            search_keyword=search_keyword,
            wait_timeout=wait_timeout,
        )

        print(f'검색 결과 URL : {source_url}')
        print()

        news_df, collection_summary = collect_news_with_infinite_scroll(
            driver=driver,
            search_keyword=search_keyword,
            batch_id=batch_id,
            max_articles=max_articles,
            max_scrolls=max_scrolls,
            max_idle_scrolls=max_idle_scrolls,
            scroll_wait_timeout=scroll_wait_timeout,
        )

        news_df = download_news_images(
            news_df=news_df,
            image_dir=image_dir,
            project_dir=PROJECT_DIR,
        )

        save_news_csv(
            news_df=news_df,
            csv_file=csv_file,
        )

        verify_saved_csv(
            csv_file=csv_file,
            original_df=news_df,
        )

    finally:
        driver.quit()

    downloaded_count = int(news_df['image_downloaded'].sum())

    print()
    print('=' * 70)
    print('네이버 뉴스 무한 스크롤 수집 완료')
    print('=' * 70)
    print(f'검색어 : {search_keyword}')
    print(f'기사 수 : {len(news_df)}')
    print(f'스크롤 수 : {collection_summary["scroll_count"]}')
    print(f'종료 이유 : {collection_summary["stop_reason"]}')
    print(f'이미지 저장 수 : {downloaded_count}')
    print(f'CSV 파일 : {csv_file}')
    print(f'이미지 폴더 : {image_dir}')

    return news_df, csv_file, image_dir, collection_summary

# 실행

In [16]:
news_df, csv_file, image_dir, collection_summary = run_naver_news_collection(
    search_keyword=SEARCH_KEYWORD,
    max_articles=MAX_ARTICLES,
    max_scrolls=MAX_SCROLLS,
    max_idle_scrolls=MAX_IDLE_SCROLLS,
    headless=True,
)

검색 결과 URL : https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query=AI

[scroll=00] 신규  10건 / 누적  10건
[scroll=01] 신규   9건 / 누적  19건
[scroll=02] 신규  10건 / 누적  29건
[scroll=03] 신규  10건 / 누적  39건
[scroll=04] 신규  10건 / 누적  49건
[scroll=05] 신규  10건 / 누적  59건
[scroll=06] 신규  10건 / 누적  69건
[scroll=07] 신규  10건 / 누적  79건
[scroll=08] 신규  10건 / 누적  89건
[scroll=09] 신규  10건 / 누적  99건
[scroll=10] 신규   1건 / 누적 100건

네이버 뉴스 무한 스크롤 수집 완료
검색어 : AI
기사 수 : 100
스크롤 수 : 10
종료 이유 : 목표 기사 수 100건 도달
이미지 저장 수 : 100
CSV 파일 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_095603\naver_news_AI_20260818_095603.csv
이미지 폴더 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260818_095603\images


# 결과 확인

In [14]:
news_df.head()

,rank,scroll_round,batch_id,search_keyword,press,title,summary,article_url,image_url,image_source_url,source_site,source_url,collected_at,image_file,image_downloaded
0,1,0,20260814_104208,AI,뉴시스,"LG 구광모 회장, 젠슨 황과 두 달 만에 재회…""로봇·AI 팩토리 협력 논...",구광모 LG그룹 회장이 미국 실리콘밸리를 찾아 젠슨 황 엔비디아 최고경영자(CEO)...,https://www.newsis.com/view/NISX20260813_00037...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/003/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-14T10:42:12,data\dynamic\naver\AI\20260814_104208\images\0...,True
1,2,0,20260814_104208,AI,연합뉴스,"삼성전자 반도체, 딥러닝·비전 AI 전문가 영입해 AX 가속화",삼성전자 반도체가 인공지능(AI) 관련 전문가를 영입해 AX(인공지능 전환) 역량 ...,https://www.yna.co.kr/view/AKR2026081403160000...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/001/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-14T10:42:12,data\dynamic\naver\AI\20260814_104208\images\0...,True
2,3,0,20260814_104208,AI,KBS,[단독] 국가시험도 줄줄이 적발…‘AI 안경’ 부정행위 4건 확인,[리포트] 안경을 쓰고 토익 문제를 바라보자 3초 만에 답을 내놓는 AI 안경. [...,https://news.kbs.co.kr/news/pc/view/view.do?nc...,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/056/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-14T10:42:12,data\dynamic\naver\AI\20260814_104208\images\0...,True
3,4,0,20260814_104208,AI,한국경제,"MSCI, AI 반도체 담고 바이오·콘텐츠 뺐다",꺾이지 않는 인공지능(AI) 인프라 투자 사이클이 각국 증시의 벤치마크까지 바꿔놓고...,https://www.hankyung.com/article/2026081301041,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/015/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-14T10:42:12,data\dynamic\naver\AI\20260814_104208\images\0...,True
4,5,0,20260814_104208,AI,경향신문,“말만 공급망 AX 전환”?…협력사 디지털 문맹에 막힌 대기업들의 상...,13일 한국경제인협회(한경협)가 매출액 상위 800대 제조업체를 대상으로 조사한 ‘...,https://www.khan.co.kr/article/202608131035001,https://search.pstatic.net/common/?src=https%3...,https://imgnews.pstatic.net/image/origin/032/2...,Naver News Search,https://search.naver.com/search.naver?ssc=tab....,2026-08-14T10:42:12,data\dynamic\naver\AI\20260814_104208\images\0...,True


In [15]:
print(f'수집 기사 수 : {len(news_df)}')
print(f'이미지 저장 수 : {int(news_df["image_downloaded"].sum())}')
print(f'종료 이유 : {collection_summary["stop_reason"]}')
print(f'CSV 파일 : {csv_file}')
print(f'이미지 폴더 : {image_dir}')

수집 기사 수 : 100
이미지 저장 수 : 100
종료 이유 : 목표 기사 수 100건 도달
CSV 파일 : D:\AI\data_analytics\alone\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260814_104208\naver_news_AI_20260814_104208.csv
이미지 폴더 : D:\AI\data_analytics\alone\crawling\01-data-collection-pipeline\data\dynamic\naver\AI\20260814_104208\images


# 권장 종료 전략

실무에서는 **"끝까지"보다 목적에 맞는 목표 수집량**을 먼저 정하는 것이 좋다.

현재 실습 권장값:

```python
MAX_ARTICLES = 100
MAX_SCROLLS = 30
MAX_IDLE_SCROLLS = 3
SCROLL_WAIT_TIMEOUT = 5
```

우선순위:

```text
1. 목표 기사 수 도달
2. 연속 3회 새 기사 없음
3. 최대 스크롤 횟수 도달
```

전체 데이터가 반드시 필요한 별도 요구가 있을 때만 `MAX_ARTICLES = None`으로 설정한다.
이 경우에도 `MAX_SCROLLS`와 `MAX_IDLE_SCROLLS` 안전장치는 유지한다.
